# Prompt Template: Responder (Raw (No Personality))

Prompt construction notebook for the responder role using raw (no personality) prompts.

**Related text file:** `_responder_raw.txt`

In [2]:
import json
import logging
import sys
import time
import subprocess
import threading
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, Any, List

import pandas as pd
import re
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor, as_completed

import langchain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

# =========================
# CONFIG
# =========================
MODEL_NAME = "llama2-uncensored:7b"  # change to your Ollama model
TEMPERATURE = 0.2
RUNS = 1000             # number of independent responder decisions to collect
MAX_WORKERS = 1         # concurrency level
INVOKE_TIMEOUT = 240    # per-call timeout seconds
MAX_RETRIES = 2         # additional retry attempts
BASE_BACKOFF = 1.0      # backoff base seconds
RESTART_THROTTLE_SEC = 30

# File paths (responder-specific)
PROMPT_PATH = "_responder_raw.txt"  # should NOT require personality variables anymore
OUTPUT_CSV = f"_data_responder_{MODEL_NAME.replace(':', '_').replace('-', '_')}_nopersonality_temp{TEMPERATURE}_{RUNS}.csv"
MALFORMED_RESPONSES_LOG = "malformed_responses_samples_responder_raw.jsonl"
RUN_LOG_FILE = "run_responder_raw.log"

# =========================
# LOGGING
# =========================
console_handler = logging.StreamHandler(sys.stdout)
file_handler = logging.FileHandler(RUN_LOG_FILE, encoding="utf-8")
fmt = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
console_handler.setFormatter(fmt)
file_handler.setFormatter(fmt)

base_logger = logging.getLogger("sim")
base_logger.setLevel(logging.INFO)
if not base_logger.handlers:
    base_logger.addHandler(console_handler)
    base_logger.addHandler(file_handler)

langchain.verbose = True

class ContextAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        extra = self.extra.copy()
        parts = [f"{k}={v}" for k, v in extra.items() if v is not None]
        prefix = "[" + " ".join(parts) + "] " if parts else ""
        return prefix + msg, kwargs

def make_logger(agent_id=None, retry=None):
    return ContextAdapter(base_logger, {"agent_id": agent_id, "retry": retry})

# =========================
# OLLAMA PROCESS MGMT (Windows)
# =========================
ollama_restarting_event = threading.Event()
restart_lock = threading.Lock()
_last_restart_time = 0.0

def is_ollama_running() -> bool:
    try:
        output = subprocess.check_output(["tasklist", "/FI", "IMAGENAME eq ollama.exe"], stderr=subprocess.STDOUT)
        return b"ollama.exe" in output
    except Exception as e:
        base_logger.error(f"Error checking Ollama process: {e}")
        return False

def restart_ollama():
    global _last_restart_time
    with restart_lock:
        now = time.time()
        if now - _last_restart_time < RESTART_THROTTLE_SEC:
            base_logger.info("Restart throttled; skipping.")
            return
        _last_restart_time = now

        base_logger.info("Restarting Ollama...")
        try:
            subprocess.run(["taskkill", "/F", "/IM", "ollama.exe"], stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                           check=True)
        except Exception as e:
            base_logger.warning(f"Failed to kill Ollama cleanly: {e}")

        wait_time = 0
        while is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to shutdown...")
            time.sleep(2)
            wait_time += 2
        if is_ollama_running():
            base_logger.error("Ollama did not terminate in time.")
        else:
            base_logger.info("Ollama terminated.")

        try:
            subprocess.Popen(["ollama", "start"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except Exception as e:
            base_logger.error(f"Error starting Ollama: {e}")

        wait_time = 0
        while not is_ollama_running() and wait_time < 20:
            base_logger.info("Waiting for Ollama to start...")
            time.sleep(2)
            wait_time += 2
        if not is_ollama_running():
            base_logger.error("Ollama did not start in time.")
        else:
            base_logger.info("Ollama restarted successfully.")

def wait_for_ollama():
    while ollama_restarting_event.is_set():
        time.sleep(0.5)

# =========================
# DATA MODELS
# =========================
@dataclass
class Agent:
    ID: int
    Token: str

# =========================
# HELPERS
# =========================
def load_prompt(file_path: str) -> str:
    return Path(file_path).read_text(encoding="utf-8")

def sanitize_prompt(text: str) -> str:
    """
    Removes any personality section and residual {d_value}/{d_description} tokens.
    Looks for a block starting with '## Your Personality and Profile' up to the next '### ' header.
    """
    # Drop the personality block if present
    text = re.sub(
        r"##\s*Your Personality and Profile.*?(?=^###\s|\Z)",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL | re.MULTILINE,
    )
    # Remove lines that directly mention D-factor scaffolding
    text = re.sub(r".*\{d_value\}.*\n?", "", text)
    text = re.sub(r".*\{d_description\}.*\n?", "", text)
    # As a last resort, strip any dangling braces placeholders named d_value/d_description
    text = text.replace("{d_value}", "").replace("{d_description}", "")
    # Clean up extra blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def normalize_decision(token: str) -> Optional[str]:
    if not token:
        return None
    t = token.strip().upper()
    accept_set = {"ACCEPT", "A", "YES", "Y", "TAKE", "ACCEPTED", "APPROVE", "APPROVAL"}
    reject_set = {"REJECT", "B", "NO", "N", "DECLINE", "REJECTED", "REFUSE", "REFUSAL", "DENY", "DENIED"}
    if t in accept_set:
        return "ACCEPT"
    if t in reject_set:
        return "REJECT"
    return None

def save_malformed_sample(agent_id, raw_response, parsed_decision, parsed_justification):
    sample = {
        "timestamp": time.time(),
        "agent_id": agent_id,
        "raw_response": raw_response,
        "parsed_decision": parsed_decision,
        "parsed_justification": parsed_justification
    }
    with open(MALFORMED_RESPONSES_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

def parse_response(response: str) -> Tuple[Optional[str], Optional[str]]:
    try:
        stripped = response.strip()

        # JSON mode
        if stripped.startswith("{"):
            obj = json.loads(stripped)
            decision = normalize_decision(obj.get("decision", ""))
            justification = obj.get("justification", "").strip()
            if decision and justification:
                return decision, justification

        # "Decision: Accept/Reject" + "Justification: ..."
        m_dec = re.search(r'Decision:\s*(ACCEPT|REJECT)', response, re.IGNORECASE)
        m_jus = re.search(r'Justification:\s*(.+)', response, re.DOTALL | re.IGNORECASE)
        if m_dec and m_jus:
            decision = normalize_decision(m_dec.group(1))
            justification = m_jus.group(1).strip()
            if decision and justification:
                return decision, justification

        # Fallback to A/B
        m_dec_ab = re.search(r'Decision:\s*([ABab])', response)
        if m_dec_ab and m_jus:
            decision = normalize_decision(m_dec_ab.group(1))
            justification = m_jus.group(1).strip()
            if decision and justification:
                return decision, justification
    except Exception:
        pass
    return None, None

def build_template(prompt_text: str) -> ChatPromptTemplate:
    sanitized = sanitize_prompt(prompt_text)
    template = ChatPromptTemplate.from_template(sanitized)
    # If template somehow still expects d_* variables, set them empty so runs don't fail
    needs = {"d_value", "d_description"}
    if any(v in getattr(template, "input_variables", []) for v in needs):
        template = template.partial(d_value="", d_description="")
    return template

# =========================
# LLM CHAIN
# =========================
llm = OllamaLLM(model=MODEL_NAME, temperature=TEMPERATURE, num_predict=4096)
prompt_text = load_prompt(PROMPT_PATH)
template = build_template(prompt_text)
responder_chain = template | llm | StrOutputParser()

def invoke_with_retries(payload: Dict[str, Any], context: Dict[str, Any]) -> Tuple[str, Optional[str], Optional[str]]:
    last_exc = None
    for attempt in range(1, MAX_RETRIES + 2):
        log = make_logger(agent_id=context.get("agent_id"), retry=attempt)
        log.info("Invoke attempt starting.")
        try:
            wait_for_ollama()
            with ThreadPoolExecutor(max_workers=1) as exec_single:
                future = exec_single.submit(lambda: responder_chain.invoke(payload))
                raw = future.result(timeout=INVOKE_TIMEOUT)
            decision, justification = parse_response(raw)
            if decision and justification:
                log.info(f"Invoke attempt success. Parsed decision={decision}.")
                return raw, decision, justification
            log.warning("Malformed output; saving sample.")
            save_malformed_sample(context.get("agent_id"), raw, decision, justification)
            if attempt > MAX_RETRIES:
                return raw, decision, justification
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s.")
            time.sleep(backoff)
        except concurrent.futures.TimeoutError:
            log.error("Timeout on invoke.")
            last_exc = "timeout"
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s after timeout.")
            time.sleep(backoff)
        except Exception as e:
            log.error(f"Unexpected error: {e}")
            last_exc = e
            if attempt == 1:
                ollama_restarting_event.set()
                restart_ollama()
                ollama_restarting_event.clear()
            backoff = BASE_BACKOFF * (2 ** (attempt - 1)) + random.uniform(0, 0.5)
            log.info(f"Retry backoff {backoff:.2f}s after exception.")
            time.sleep(backoff)
    return f"Error after retries: {last_exc}", None, None

# =========================
# PIPELINE
# =========================
def process_responder(agent: Agent) -> Dict[str, Any]:
    """
    One decision per agent. No personality inputs; prompt is fixed.
    """
    log = make_logger(agent_id=agent.ID)
    start = time.time()
    log.info(f"AGENT_START token={agent.Token}")

    context = {"agent_id": agent.ID}
    payload: Dict[str, Any] = {}  # nothing to pass; prompt is fixed
    raw_resp, decision, justification = invoke_with_retries(payload, context)
    if decision is None or justification is None:
        log.warning(f"Final parse failure. decision={decision} justification={justification}")

    dur = time.time() - start
    log.info(f"AGENT_END token={agent.Token} duration_sec={dur:.2f} decision={decision}")

    return {
        "agent_id": agent.ID,
        "token": agent.Token,
        "q1_full_response": raw_resp,
        "q1_decision": decision,
        "q1_text": justification,
    }

def simulate_responses_responder(agents_df: pd.DataFrame) -> pd.DataFrame:
    agents = [Agent(ID=row["ID"], Token=row["Token"]) for _, row in agents_df.iterrows()]
    results = []
    total = len(agents)
    base_logger.info(f"Submitting {total} agents with MAX_WORKERS={MAX_WORKERS}...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {}
        for ag in agents:
            base_logger.info(f"SUBMIT agent_id={ag.ID} token={ag.Token}")
            fut = executor.submit(process_responder, ag)
            future_map[fut] = ag

        for i, future in enumerate(as_completed(future_map), 1):
            agent = future_map[future]
            try:
                res = future.result()
                results.append(res)
                base_logger.info(f"COMPLETE agent_id={agent.ID} token={agent.Token} ({i}/{total})")
            except Exception as exc:
                make_logger(agent_id=agent.ID).error(f"Agent crashed: {exc}")
            if i % max(1, total // 4) == 0 or i == total:
                base_logger.info(f"Progress {i}/{total} agents ({(i / total) * 100:.1f}%)")
    return pd.DataFrame(results)

# =========================
# MAIN
# =========================
if __name__ == "__main__":
    # Build N simple agents (no traits)
    agents_df = pd.DataFrame({
        "ID": list(range(1, RUNS + 1)),
        "Token": [f"R{i}" for i in range(1, RUNS + 1)]
    })

    # Run the simulation (single fixed decision per agent)
    simulation_results = simulate_responses_responder(agents_df)
    simulation_results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    base_logger.info(f"Wrote results to {OUTPUT_CSV}")
    print(simulation_results.head())


2025-08-25 23:06:57,056 - INFO - Submitting 1000 agents with MAX_WORKERS=1...
2025-08-25 23:06:57,056 - INFO - SUBMIT agent_id=1 token=R1
2025-08-25 23:06:57,057 - INFO - [agent_id=1] AGENT_START token=R1
2025-08-25 23:06:57,057 - INFO - SUBMIT agent_id=2 token=R2
2025-08-25 23:06:57,059 - INFO - [agent_id=1 retry=1] Invoke attempt starting.
2025-08-25 23:06:57,059 - INFO - SUBMIT agent_id=3 token=R3
2025-08-25 23:06:57,060 - INFO - SUBMIT agent_id=4 token=R4
2025-08-25 23:06:57,062 - INFO - SUBMIT agent_id=5 token=R5
2025-08-25 23:06:57,062 - INFO - SUBMIT agent_id=6 token=R6
2025-08-25 23:06:57,063 - INFO - SUBMIT agent_id=7 token=R7
2025-08-25 23:06:57,063 - INFO - SUBMIT agent_id=8 token=R8
2025-08-25 23:06:57,064 - INFO - SUBMIT agent_id=9 token=R9
2025-08-25 23:06:57,064 - INFO - SUBMIT agent_id=10 token=R10
2025-08-25 23:06:57,064 - INFO - SUBMIT agent_id=11 token=R11
2025-08-25 23:06:57,065 - INFO - SUBMIT agent_id=12 token=R12
2025-08-25 23:06:57,065 - INFO - SUBMIT agent_id=1